# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [89]:
## Environment Setup

# ============================================================
# Orbis openWakeWord Training Bootstrap / Patches (Colab v3)
# ============================================================
# Purpose:
# - Recreate a working training environment despite dependency/API drift
# - Apply all patch fixes discovered during setup
#
# Safe to rerun in a fresh Colab runtime.
# ============================================================

import os
import sys
from pathlib import Path
import subprocess
import textwrap

def sh(cmd: str, optional: bool = False):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        if optional:
            print(f"WARNING: optional command failed ({result.returncode}): {cmd}")
            return False
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return True

# -----------------------------
# 1) System deps
# -----------------------------
sh("apt-get update -y")
sh("apt-get install -y espeak-ng libespeak-ng1")

# -----------------------------
# 2) Python deps (training env)
# -----------------------------
# Core notebook deps + compatibility deps
# Some packages are flaky / optional in Colab; mark them optional.
pip_installs_required = [
    "piper-tts",
    "webrtcvad",
    "mutagen==1.47.0",
    "torchinfo==1.8.0",
    "torchmetrics==1.2.0",
    "speechbrain==0.5.14",
    "audiomentations==0.33.0",
    "torch-audiomentations==0.11.0",
    "acoustics==0.2.6",
    "pronouncing==0.2.0",
    "datasets==2.14.6",
    "deep-phonemizer==0.0.19",
    "espeak-phonemizer",
    "onnx",
    "onnxscript",
]

pip_installs_optional = [
    "piper-phonemize",
    # Optional TFLite conversion stack (often incompatible with current Colab Python)
    "tensorflow-cpu==2.8.1",
    "tensorflow_probability==0.16.0",
    "onnx_tf==1.10.0",
]

for pkg in pip_installs_required:
    sh(f"{sys.executable} -m pip install -q {pkg}")

for pkg in pip_installs_optional:
    sh(f"{sys.executable} -m pip install -q {pkg}", optional=True)

# -----------------------------
# 3) Clone repos (clean/fixed versions)
# -----------------------------
# piper-sample-generator: use dscripka fork expected by the notebook flow
if Path("/content/piper-sample-generator").exists():
    sh("rm -rf /content/piper-sample-generator")
sh("git clone https://github.com/dscripka/piper-sample-generator /content/piper-sample-generator")

# Download Piper voice model used by generator
os.makedirs("/content/piper-sample-generator/models", exist_ok=True)
sh(
    "wget -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt "
    "'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'"
)

# The generator expects this legacy filename; create symlink
sh(
    "ln -sf /content/piper-sample-generator/models/en_US-libritts_r-medium.pt "
    "/content/piper-sample-generator/models/en-us-libritts-high.pt"
)

# openWakeWord repo (training scripts + examples)
if Path("/content/openwakeword").exists():
    sh("rm -rf /content/openwakeword")
sh("git clone https://github.com/dscripka/openwakeword /content/openwakeword")
sh(f"{sys.executable} -m pip install -q -e /content/openwakeword")

# -----------------------------
# 4) Download openWakeWord required resource models (Colab workaround)
# -----------------------------
os.makedirs("/content/openwakeword/openwakeword/resources/models", exist_ok=True)
sh("wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O /content/openwakeword/openwakeword/resources/models/embedding_model.onnx")
sh("wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O /content/openwakeword/openwakeword/resources/models/embedding_model.tflite")
sh("wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O /content/openwakeword/openwakeword/resources/models/melspectrogram.onnx")
sh("wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O /content/openwakeword/openwakeword/resources/models/melspectrogram.tflite")

# -----------------------------
# 5) Patch torch_audiomentations for modern torchaudio
#    - torchaudio.set_audio_backend may not exist
#    - torchaudio.info may not exist
# -----------------------------
ta_io = Path("/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py")
if ta_io.exists():
    src = ta_io.read_text()

    # Repair old buggy bootstrap insertion (literal \n in file content)
    src = src.replace(
        '\\n\\nif hasattr(torchaudio, "set_audio_backend"):',
        '\n\nif hasattr(torchaudio, "set_audio_backend"):'
    )

    # Ensure imports exist
    if "import soundfile as sf" not in src:
        src = src.replace(
            "import torchaudio",
            "import torchaudio\nimport soundfile as sf\nfrom types import SimpleNamespace"
        )

    # Patch deprecated backend call safely
    old_backend_line = 'torchaudio.set_audio_backend("soundfile")'
    if old_backend_line in src:
        src = src.replace(
            old_backend_line,
            'if hasattr(torchaudio, "set_audio_backend"):\n    torchaudio.set_audio_backend("soundfile")'
        )

    # Add torchaudio.info fallback (with REAL newlines)
    if "torchaudio.info = _ta_info_fallback" not in src:
        fallback = textwrap.dedent("""
        if not hasattr(torchaudio, "info"):
            def _ta_info_fallback(file_path):
                i = sf.info(file_path)
                return SimpleNamespace(num_frames=i.frames, sample_rate=i.samplerate)
            torchaudio.info = _ta_info_fallback
        """).strip() + "\n"

        marker = 'if hasattr(torchaudio, "set_audio_backend"):'
        if marker in src:
            idx = src.find(marker)
            src = src[:idx] + fallback + "\n" + src[idx:]
        else:
            src = fallback + "\n" + src

    ta_io.write_text(src)
    print(f"Patched torch_audiomentations IO compatibility: {ta_io}")
else:
    print("WARNING: torch_audiomentations io.py not found; patch skipped")

# -----------------------------
# 6) Patch piper-sample-generator for PyTorch 2.6+ torch.load default change
# -----------------------------
piper_gen = Path("/content/piper-sample-generator/generate_samples.py")
if piper_gen.exists():
    src = piper_gen.read_text()
    old = "model = torch.load(model_path)"
    new = "model = torch.load(model_path, weights_only=False)"
    if old in src and new not in src:
        src = src.replace(old, new)
        piper_gen.write_text(src)
        print(f"Patched piper-sample-generator torch.load weights_only=False: {piper_gen}")
    else:
        print("piper-sample-generator torch.load patch already applied or target line not found")
else:
    print("WARNING: piper generate_samples.py not found; patch skipped")

# -----------------------------
# 7) Patch deep-phonemizer for PyTorch 2.6+ torch.load default change
# -----------------------------
dp_model = Path("/usr/local/lib/python3.12/dist-packages/dp/model/model.py")
if dp_model.exists():
    src = dp_model.read_text()
    old = "checkpoint = torch.load(checkpoint_path, map_location=device)"
    new = "checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)"
    if old in src and new not in src:
        src = src.replace(old, new)
        dp_model.write_text(src)
        print(f"Patched deep-phonemizer torch.load weights_only=False: {dp_model}")
    else:
        print("deep-phonemizer torch.load patch already applied or target line not found")
else:
    print("WARNING: deep-phonemizer model.py not found; patch skipped")

# -----------------------------
# 8) Quick verification
# -----------------------------
print("\nBootstrap complete.")
print("Python:", sys.version)
print("openWakeWord repo exists:", Path("/content/openwakeword").exists())
print("piper-sample-generator repo exists:", Path("/content/piper-sample-generator").exists())
print("Piper model symlink exists:", Path("/content/piper-sample-generator/models/en-us-libritts-high.pt").exists())
print("OWW resource models dir exists:", Path("/content/openwakeword/openwakeword/resources/models").exists())

# Handy checks (uncomment if needed)
# !grep -n "set_audio_backend\\|torchaudio.info = _ta_info_fallback" /usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py
# !grep -n "torch.load(model_path" /content/piper-sample-generator/generate_samples.py
# !grep -n "torch.load(checkpoint_path" /usr/local/lib/python3.12/dist-packages/dp/model/model.py


$ apt-get update -y
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
$ apt-get install -y espeak-ng libespeak-ng1
Reading package lists...
Building dependency tree...
Reading state information...
libespeak-ng1 is already the newest version (1.50+dfsg-10ubuntu0.1).
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 104 not upgraded.
$ /usr/bin/python3 -m pip install -q pi

In [90]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [91]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [03:59,  1.13it/s]


In [92]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


--2026-02-27 02:20:23--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 3.169.137.111, 3.169.137.5, 3.169.137.19, ...
Connecting to huggingface.co (huggingface.co)|3.169.137.111|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-02-27 02:20:23 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]
 99%|█████████▉| 119/120 [00:57<00:00,  2.08it/s]


In [93]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-02-27 02:22:08--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 3.169.137.119, 3.169.137.19, 3.169.137.5, ...
Connecting to huggingface.co (huggingface.co)|3.169.137.119|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&Expires=1772162528&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcyMTYyNTI4fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjU

In [94]:
TOKEN_PHRASE = "domus"
MODEL_NAME = "domus"   # or TOKEN_PHRASE.replace(" ", "_")
CANONICAL_TOKEN = "domus"  # sidecar label mapping later


# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase in `TOKEN_PHRASE` (e.g., "domus")
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.safe_load(open("openwakeword/examples/custom_model.yml", 'r').read())
config


In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = [TOKEN_PHRASE]
config["model_name"] = MODEL_NAME

config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.safe_dump(config, file, sort_keys=False)


In [105]:
assert config["model_name"] == MODEL_NAME
assert config["target_phrase"] == [TOKEN_PHRASE]


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [107]:
# Reset broken audio augmentation stack
!pip -q uninstall -y torch-audiomentations torch_audiomentations julius torch-pitch-shift

# Install a known-good combo
!pip -q install "torch-audiomentations==0.11.1" "julius==0.2.7" "torch-pitch-shift==1.2.5"

# Optional: restart runtime after this if imports still look weird


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.6 MB/s eta 0:00:00


In [108]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
INFO:root:##################################################
Generating negative clips for testing
##################################################


In [109]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [110]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 883, in <module>
    X_val_neg = np.load(os.path.join(feature_save_dir, "negative_features_test.npy"))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 427, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/my_custom_model/domus/negative_features_test.npy'


In [ ]:
# Verify outputs
model_name = config["model_name"]
!find /content/my_custom_model -type f | egrep "{model_name}|\.onnx$|\.tflite$" | sort


In [101]:
!pip -q install -U onnx2tf


In [ ]:
import glob

model_name = config["model_name"]
onnx_candidates = sorted(glob.glob(f"/content/my_custom_model/**/{model_name}.onnx", recursive=True))
assert onnx_candidates, f"No ONNX found for {model_name} under /content/my_custom_model"
onnx_path = onnx_candidates[-1]
out_dir = "/content/my_custom_model/tflite_export"

!onnx2tf -i "{onnx_path}" -o "{out_dir}"
!find "{out_dir}" -type f | egrep "\.tflite$" | sort


In [103]:
!ls -lh /content/my_custom_model/tflite_export/*.tflite


-rw-r--r-- 1 root root 106K Feb 26 23:51 /content/my_custom_model/tflite_export/orbis_float16.tflite
-rw-r--r-- 1 root root 203K Feb 26 23:51 /content/my_custom_model/tflite_export/orbis_float32.tflite
-rw-r--r-- 1 root root 106K Feb 27 01:44 /content/my_custom_model/tflite_export/orbus_float16.tflite
-rw-r--r-- 1 root root 203K Feb 27 01:44 /content/my_custom_model/tflite_export/orbus_float32.tflite


In [ ]:
import os, glob, shutil, subprocess
from google.colab import files

model_name = config["model_name"]
onnx_candidates = sorted(glob.glob(f"/content/my_custom_model/**/{model_name}.onnx", recursive=True))
assert onnx_candidates, f"No ONNX found for {model_name} under /content/my_custom_model"
onnx_path = onnx_candidates[-1]

out_dir = "/content/my_custom_model/tflite_export"
os.makedirs(out_dir, exist_ok=True)

# Ensure converter installed
subprocess.run(["pip", "-q", "install", "-U", "onnx2tf"], check=True)

# Convert ONNX -> TFLite
subprocess.run(["onnx2tf", "-i", onnx_path, "-o", out_dir], check=True)

# Pick float32 if present, else fallback to first tflite
float32_candidates = sorted(glob.glob(f"{out_dir}/{model_name}*_float32.tflite"))
all_tflites = sorted(glob.glob(f"{out_dir}/*.tflite"))
assert all_tflites, f"No .tflite generated in {out_dir}"

if float32_candidates:
    tflite_path = float32_candidates[-1]
else:
    tflite_path = f"{out_dir}/{model_name}_float32.tflite"
    shutil.copy2(all_tflites[0], tflite_path)

# Verify artifacts
subprocess.run(["bash", "-lc", f"ls -lh '{onnx_path}' '{tflite_path}'"], check=True)


In [ ]:
import os, glob, zipfile, shutil
from google.colab import files

model_name = config["model_name"]
onnx_candidates = sorted(glob.glob(f"/content/my_custom_model/**/{model_name}.onnx", recursive=True))
assert onnx_candidates, f"No ONNX found for {model_name} under /content/my_custom_model"
onnx_path = onnx_candidates[-1]

out_dir = "/content/my_custom_model/tflite_export"
float32_candidates = sorted(glob.glob(f"{out_dir}/{model_name}*_float32.tflite"))
all_tflites = sorted(glob.glob(f"{out_dir}/*.tflite"))
assert all_tflites, f"No .tflite generated in {out_dir}"

if float32_candidates:
    tflite_path = float32_candidates[-1]
else:
    tflite_path = f"{out_dir}/{model_name}_float32.tflite"
    shutil.copy2(all_tflites[0], tflite_path)

zip_path = f"/content/{model_name}_model_bundle.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(onnx_path, arcname=f"{model_name}.onnx")
    z.write(tflite_path, arcname=f"{model_name}_float32.tflite")

files.download(zip_path)
